In [1]:
##### Creates rasters of crop production value 
### Takes FAO country-level production value stats and apportions using MapSPAM production volume rasters 
# unit = $

import os
import pandas as pd
import geopandas as gpd
import rioxarray as rio
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from glob import glob
import rasterio
from rasterio.warp import reproject, Resampling
from matplotlib.colors import BoundaryNorm
import matplotlib.colors as mcolors
from rasterio.features import rasterize

In [3]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# load country data 
crop_value = pd.read_csv(f"{cd}/Data/Clean/Production/FAO_data/FAO_production_value_clean_commodity.csv")
country_shp = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/gadm_410-levels.gpkg", layer='ADM_0')

commodities = pd.read_csv(f"{cd}/Data/Correspondence_tables/FAO_commodity_groups.csv")

In [57]:
#### Clean crop_value data

# drop livestock commodities 
livestock = ['all', 'cattle', 'chicken', 'goats', 'pigs', 'sheep','horses', 'ducks', 'buffalo']
crop_value = crop_value[~crop_value['final_group'].isin(livestock)]

# add commodity codes
crop_value = crop_value.merge(commodities, on='final_group', how='left')

# convert filenames to caps
crop_value['final_group'] = crop_value['final_group'].str.upper()

# convert to $ 
crop_value['ag_production_USD_2020'] = crop_value['ag_production_thousand_USD_2020'] * 1e3

# convert to wide 
crop_value_wide = crop_value.pivot_table(index='ISO3', columns='final_group', values='ag_production_USD_2020')
crop_value_wide = crop_value_wide.reset_index()

# fill missing with 0's
crop_value_wide = crop_value_wide.fillna(0)

In [58]:
##### Create aggregate rasters for coffee and millet
# agg_coffee, agg_millet 

# create aggregate coffee raster
with rasterio.open(f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_COFF_A.tif") as src1:
    data1 = src1.read(1)
    profile = src1.profile

with rasterio.open(f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_RCOF_A.tif") as src2:
    data2 = src2.read(1)

result = data1 + data2

with rasterio.open(f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_agg_coffee_A.tif", 'w', **profile) as dst:
    dst.write(result, 1)

# create aggregate millet raster
with rasterio.open(f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_MILL_A.tif") as src1:
    data1 = src1.read(1)
    profile = src1.profile

with rasterio.open(f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_PMIL_A.tif") as src2:
    data2 = src2.read(1)

result = data1 + data2

with rasterio.open(f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_agg_millet_A.tif", 'w', **profile) as dst:
    dst.write(result, 1)

In [59]:
##### Define function to distribute country crop production (value) by crop production (weight)

def apportion_country_production(crop_type, country_shp):

    ### Define
    raster_path = f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_{crop_type}_A.tif"
    output_path = f"{cd}/Data/Clean/Production/crop_value/{crop_type}_crop_value_USD_2020.tif"
    column_title = crop_type

    ### Set-up 
    # create dictionary of countries 
    crop_dict = dict(zip(crop_value_wide['ISO3'], crop_value_wide[column_title]))

    # load raster 
    with rasterio.open(raster_path) as src:
        crop_weight = src.read(1).astype(np.float64)  
        nodata      = src.nodata
        meta        = src.meta.copy()
        transform   = src.transform
        crs         = src.crs
        height, width = crop_weight.shape

    # mask nodata and negative values
    if nodata is not None:
        crop_weight[crop_weight == nodata] = np.nan
    crop_weight[crop_weight < 0] = np.nan

    # ensure crs's match
    country_shp = country_shp.to_crs(crs)

    # set output shape
    output = np.full((height, width), np.nan, dtype=np.float64)

    # iterate through each country, creating a mask and reapportioning the country total based on production weight 
    for _, row in country_shp.iterrows():
        iso3  = str(row['GID_0']).upper()
        crops = crop_dict.get(iso3, None)

        if pd.isna(crops) or crops is None:
            continue   

        # rasterize country mask
        mask = rasterize(
            [(row.geometry, 1)],
            out_shape=(height, width),
            transform=transform,
            fill=0,
            dtype=np.uint8
        ).astype(bool)

        total_crops = np.nansum(crop_weight[mask])

        if total_crops <= 0:
            continue
        
        output[mask] = (crop_weight[mask] / total_crops) * crops

    ### Write output 
    meta.update(
        dtype   = "float32",
        count   = 1,
        nodata  = -9999,
        compress= "lzw"
    )

    out_arr = output.astype(np.float32)
    out_arr[np.isnan(out_arr)] = -9999

    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(out_arr, 1)

In [60]:
##### Run country appotionment function for crops

crop_types = ['WHEA', 'RICE', 'BARL', 'MAIZ', 'OCER', 'AGG_MILLET', 'SORG',
       'COTT', 'OFIB', 'BANA', 'PLNT', 'CITR', 'TEMF', 'TROF', 'SOYB',
       'GROU', 'CNUT', 'OILP', 'OOIL', 'SUNF', 'RAPE', 'SESA', 'BEAN',
       'OPUL', 'CHIC', 'COWP', 'PIGE', 'LENT', 'REST', 'POTA', 'SWPO',
       'CASS', 'ORTS', 'YAMS', 'AGG_COFFEE', 'COCO', 'TEAS', 'TOBA',
       'SUGC', 'SUGB', 'VEGE', 'TOMA', 'ONIO', 'RUBB']

for crop_type in crop_types:
    apportion_country_production(crop_type, country_shp)

In [62]:
##### Aggregate into total crop production layer (value)

# load first raster to get metadata and shape
with rasterio.open(f"{cd}/Data/Clean/Production/Crop_value/{crop_types[0]}_crop_value_USD_2020.tif") as src:
    meta = src.meta.copy()
    nodata = src.nodata
    total = src.read(1).astype(np.float64)
    total[total == nodata] = 0
    total[total < 0] = 0

# add remaining rasters
for crop_type in crop_types[1:]:
    raster_path = f"{cd}/Data/Clean/Production/Crop_value/{crop_type}_crop_value_USD_2020.tif"
    with rasterio.open(raster_path) as src:
        data = src.read(1).astype(np.float64)
        nodata = src.nodata
        if nodata is not None:
            data[data == nodata] = 0
        data[data < 0] = 0
        total += data

# write output
meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")
out_arr = total.astype(np.float32)

with rasterio.open(f"{cd}/Data/Clean/Production/crop_production_USD_2020.tif", "w", **meta) as dst:
    dst.write(out_arr, 1)

In [63]:
##### Produce global map for crop production (weight)

# Set file paths
raster_folder = f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production"
save_path = f"{cd}/Data/Clean/Production/crop_production_tons_2020.tif"

# Get paths to all rasters in MapSPAM, skipping aggregate files
skip_files = ['spam2020_V2r0_global_P_agg_coffee_A.tif', 'spam2020_V2r0_global_P_agg_millet_A.tif']
raster_paths = [p for p in glob(os.path.join(raster_folder, "*.tif")) if os.path.basename(p) not in skip_files]
raster_paths.sort()  

sum_array = None
profile = None

# Use first raster as template
template_path = raster_paths[0]

with rasterio.open(template_path) as template_src:
    profile = template_src.profile.copy()
    template_data = template_src.read(1).astype(np.float32)

    # Initialize sum array based on template data
    sum_array = np.zeros_like(template_data, dtype=np.float32)

    # Add template raster first
    if template_src.nodata is not None:
        template_data = np.where(template_data == template_src.nodata, np.nan, template_data)
    sum_array += np.nan_to_num(template_data)

    # Loop through remaining rasters
    for path in raster_paths[1:]:
        with rasterio.open(path) as src:

            # Prepare array to match template
            resampled = np.empty_like(template_data, dtype=np.float32)

            # Reproject/resample to match template
            reproject(
                source=src.read(1),
                destination=resampled,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=template_src.transform,
                dst_crs=template_src.crs,
                resampling=Resampling.bilinear
            )

            # Handle nodata (nan_to_num ensures that nan's are treated as 0's)
            if src.nodata is not None:
                resampled = np.where(resampled == src.nodata, np.nan, resampled)

            # Add to sum
            sum_array += np.nan_to_num(resampled)

# Update profile
profile.update(dtype=rasterio.float32, nodata=np.nan, compress="lzw")

# Write output
with rasterio.open(save_path, "w", **profile) as dst:
    dst.write(sum_array, 1)